# Phase 7: Baseline Comparison: `TopologicalEGNN` vs fpocket

Every result so far compares the model to *its own ablations*. That establishes
which inputs matter, but not whether the model is any good. "Better than our
seq-only arm" is not an answer to "better than what?", which is the first
question any reviewer asks.

**fpocket** (Le Guilloux, Schmidtke & Tuffery, *BMC Bioinformatics* 2009) is the
standard geometric pocket detector: alpha-sphere Voronoi decomposition, purely
geometric, no learning. It is the right first baseline precisely because it uses
*no training data at all*, if a trained model cannot beat it, the learning is
not contributing.

### Making the comparison fair

fpocket returns **ranked pockets**, not per-residue probabilities, so a naive
comparison would be rigged. Two conversions are scored:

1. **Continuous**, each residue receives the *score of the best-ranked pocket it
   lines*, 0 if it lines none. This is rankable, so PR-AUC and ROC-AUC are
   computable and directly comparable to the model.
2. **Discrete top-N**, residues lining the top 1 / 3 pockets are positive.
   Reported as precision / recall / F1 / MCC, which is how fpocket is used in
   practice.

Both methods are evaluated on **exactly the same residues with exactly the same
labels**, and fpocket runs on the same structures the model sees.

## Cell 1: Setup and build fpocket from source

In [ ]:
!pip install -q torch-geometric==2.6.1 biopython==1.84
!apt-get -qq update > /dev/null 2>&1
!apt-get -qq install -y libnetcdf-dev build-essential > /dev/null 2>&1

In [ ]:
import gzip, os, shutil, subprocess, time, urllib.request, warnings
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
warnings.filterwarnings("ignore")

t0 = time.time()
FPOCKET = None
try:
    subprocess.run("curl -sL https://github.com/Discngine/fpocket/archive/refs/tags/4.1.tar.gz "
                   "-o /tmp/fp.tar.gz && tar xzf /tmp/fp.tar.gz -C /tmp",
                   shell=True, check=True, timeout=900)
    src = next(Path("/tmp").glob("fpocket-*"))
    subprocess.run(f"cd {src} && make -j$(nproc)", shell=True, check=True,
                   timeout=1800, capture_output=True)
    cand = src / "bin" / "fpocket"
    if cand.exists():
        os.chmod(cand, 0o755); FPOCKET = str(cand)
except Exception as e:
    print(f"fpocket build failed: {type(e).__name__}: {e}")

if FPOCKET:
    v = subprocess.run([FPOCKET], capture_output=True, text=True)
    print(f"fpocket built in {time.time()-t0:.0f}s -> {FPOCKET}")
    print((v.stdout or v.stderr).splitlines()[0][:100])
else:
    raise RuntimeError("fpocket unavailable -- cannot run the baseline")

## Cell 2: Load data and train the model on the fixed split

In [ ]:
from torch_geometric.loader import DataLoader
from sklearn.metrics import average_precision_score, matthews_corrcoef, roc_auc_score

def locate(f):
    c=[]
    for r in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if r.exists(): c.extend(r.rglob(f))
    if not c: raise FileNotFoundError(f)
    return max(c, key=lambda p: p.stat().st_size)

blob = torch.load(locate("phase6b_graphs.pt"), map_location="cpu", weights_only=False)
dataset = blob["dataset"]
TOPO_NAMES = blob["topo_names"]; TOPO_DIM = len(TOPO_NAMES)
N_SEQ = int(dataset[0].x.size(1)) - TOPO_DIM
POS_WEIGHT = float(blob["pos_weight"])

GPU_OK=False
if torch.cuda.is_available():
    try:
        _p=torch.randn(256,256,device="cuda"); (_p@_p).sum().item(); torch.cuda.synchronize(); GPU_OK=True
    except RuntimeError: pass
DEVICE = torch.device("cuda" if GPU_OK else "cpu")

splits = {s: [d for d in dataset if d.split == s] for s in ("train","val","test")}
print(f"device {DEVICE} | " + " | ".join(f"{k} {len(v)}" for k,v in splits.items()))

In [ ]:
class EGCL(nn.Module):
    def __init__(s, h, ea=2, act=nn.SiLU, uc=True):
        super().__init__(); s.uc=uc
        s.edge_mlp=nn.Sequential(nn.Linear(2*h+1+ea,h),act(),nn.Linear(h,h),act())
        s.node_mlp=nn.Sequential(nn.Linear(2*h,h),act(),nn.Linear(h,h))
        s.att=nn.Sequential(nn.Linear(h,1),nn.Sigmoid())
        if uc:
            s.coord_mlp=nn.Sequential(nn.Linear(h,h),act(),nn.Linear(h,1,bias=False))
            nn.init.xavier_uniform_(s.coord_mlp[-1].weight,gain=0.001)
        s.norm=nn.LayerNorm(h)
    def forward(s,h,x,ei,ea):
        src,dst=ei; rel=x[src]-x[dst]; d2=(rel**2).sum(-1,keepdim=True)
        m=s.edge_mlp(torch.cat([h[src],h[dst],d2,ea],-1)); m=m*s.att(m)
        agg=torch.zeros_like(h).index_add_(0,dst,m)
        deg=torch.zeros(h.size(0),1,device=h.device).index_add_(0,dst,torch.ones_like(d2)).clamp(min=1)
        h=s.norm(h+s.node_mlp(torch.cat([h,agg/deg],-1)))
        if s.uc:
            w=s.coord_mlp(m)*rel/(d2.sqrt()+1.0)
            x=x+torch.zeros_like(x).index_add_(0,dst,w)/deg
        return h,x

class TopoFusion(nn.Module):
    def __init__(s,h,t):
        super().__init__(); s.proj=nn.Linear(t,h); s.gate=nn.Sequential(nn.Linear(h+t,h),nn.Sigmoid())
    def forward(s,h,t):
        g=s.gate(torch.cat([h,t],-1)); s.last_gate=g.detach(); return h+g*s.proj(t)

class TopologicalEGNN(nn.Module):
    def __init__(s,n_seq,td,h=128,L=4,ea=2,dr=0.1):
        super().__init__(); s.n_seq=n_seq
        s.embed=nn.Sequential(nn.Linear(n_seq+td,h),nn.SiLU(),nn.Linear(h,h))
        s.layers=nn.ModuleList([EGCL(h,ea,uc=(i<L-1)) for i in range(L)])
        s.fusions=nn.ModuleList([TopoFusion(h,td) for _ in range(L)])
        s.head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Dropout(dr),nn.Linear(h//2,1))
    def forward(s,d):
        h=s.embed(d.x); pos=d.pos; t=d.x[:,s.n_seq:]
        for l,f in zip(s.layers,s.fusions):
            h,pos=l(h,pos,d.edge_index,d.edge_attr); h=f(h,t)
        return s.head(h).squeeze(-1)

def run_epoch(model, loader, opt=None):
    tr = opt is not None; model.train(tr)
    crit=nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT],device=DEVICE))
    ps,ys=[],[]
    for b in loader:
        b=b.to(DEVICE)
        with torch.set_grad_enabled(tr):
            lg=model(b); loss=crit(lg,b.y)
        if tr:
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        ps.append(torch.sigmoid(lg).detach().cpu().numpy()); ys.append(b.y.cpu().numpy())
    p,y=np.concatenate(ps),np.concatenate(ys)
    return {"pr":average_precision_score(y,p),"roc":roc_auc_score(y,p),"p":p,"y":y}

torch.manual_seed(0)
model=TopologicalEGNN(N_SEQ,TOPO_DIM).to(DEVICE)
opt=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=40)
tl=DataLoader(splits["train"],batch_size=16,shuffle=True); vl=DataLoader(splits["val"],batch_size=16)
best,state,bad=-1,None,0
t0=time.time()
for ep in range(40):
    run_epoch(model,tl,opt); v=run_epoch(model,vl); sch.step()
    if v["pr"]>best: best,state,bad=v["pr"],{k:t.clone() for k,t in model.state_dict().items()},0
    else:
        bad+=1
        if bad>=8: break
    if ep%5==0: print(f"  ep {ep:2d} val PR {v['pr']:.4f}")
model.load_state_dict(state)
print(f"trained in {(time.time()-t0)/60:.1f} min | best val PR {best:.4f}")

# per-chain model predictions on the test split
model.eval(); model_pred={}
with torch.no_grad():
    for d in splits["test"]:
        model_pred[(d.pdb_id,d.chain_id)] = torch.sigmoid(model(d.clone().to(DEVICE))).cpu().numpy()
print(f"model predictions for {len(model_pred)} test chains")

## Cell 3: Run fpocket on the same structures

Residue order is rebuilt with the identical predicate used throughout (standard
residue with a Cα), so fpocket's `(chain, resnum, icode)` keys map onto the same
indices the model predicts over. Chains whose residue count disagrees are
dropped rather than silently misaligned.

In [ ]:
from Bio.PDB import PDBParser, MMCIFParser
AA3={"ALA","ARG","ASN","ASP","CYS","GLN","GLU","GLY","HIS","ILE","LEU","LYS",
     "MET","PHE","PRO","SER","THR","TRP","TYR","VAL","MSE"}
RAW=Path("/tmp/pdb_raw"); RAW.mkdir(parents=True,exist_ok=True)
WORK=Path("/tmp/fpwork"); shutil.rmtree(WORK,ignore_errors=True); WORK.mkdir()

need=sorted({d.pdb_id for d in splits["test"]})
def fetch(pid):
    p=RAW/f"{pid}.pdb.gz"
    if p.exists() and p.stat().st_size>500: return pid,p
    try:
        with urllib.request.urlopen(f"https://files.rcsb.org/download/{pid}.pdb.gz",timeout=60) as r:
            b=r.read()
        if len(b)>500: p.write_bytes(b); return pid,p
    except Exception: pass
    return pid,None
with ThreadPoolExecutor(max_workers=16) as ex: got=list(ex.map(fetch,need))
paths={p:f for p,f in got if f}
print(f"fetched {len(paths)}/{len(need)} structures")

def run_fpocket(pid, verbose=False):
    """-> {(chain,resnum,icode): best_pocket_score}, and pocket rank -> residue set"""
    plain=WORK/f"{pid}.pdb"
    if not plain.exists():
        with gzip.open(paths[pid],"rb") as fi, open(plain,"wb") as fo: shutil.copyfileobj(fi,fo)
    outdir=WORK/f"{pid}_out"
    if not outdir.exists():
        # NOTE: do NOT use check=True. Some fpocket builds return a non-zero exit
        # status while still writing valid output, and swallowing the error makes
        # a total failure look like "0 pockets found".
        try:
            r=subprocess.run([FPOCKET,"-f",str(plain)],capture_output=True,timeout=600,text=True)
        except Exception as e:
            if verbose: print(f"  subprocess raised: {type(e).__name__}: {e}")
            return None
        if verbose:
            print(f"  returncode : {r.returncode}")
            print(f"  stdout tail: {(r.stdout or '')[-400:]!r}")
            print(f"  stderr tail: {(r.stderr or '')[-400:]!r}")
            print(f"  workdir now: {sorted(x.name for x in WORK.iterdir())[:8]}")
    if not outdir.exists():
        if verbose: print(f"  expected output dir missing: {outdir}")
        return None

    # pocket scores from the info file, residues from each pocket's atom file
    scores={}
    info=outdir/f"{pid}_info.txt"
    if info.exists():
        cur=None
        for line in info.read_text().splitlines():
            ls=line.strip()
            if ls.startswith("Pocket") and ":" in ls:
                cur=int(ls.split()[1])
            elif cur is not None and "Score" in ls and ":" in ls and "Druggability" not in ls:
                try: scores[cur]=float(ls.split(":")[1].strip())
                except ValueError: pass
    pockets={}
    for f in sorted((outdir/"pockets").glob("pocket*_atm.pdb")):
        rank=int(f.stem.split("_")[0].replace("pocket",""))
        keys=set()
        for line in f.read_text().splitlines():
            if line.startswith(("ATOM","HETATM")):
                ch=line[21]; num=line[22:26].strip(); ic=line[26].strip()
                if num: keys.add((ch,int(num),ic))
        pockets[rank]=keys
    return pockets, scores

# --- diagnose on ONE structure with full output before looping over 292 ---
probe_id=sorted(paths)[0]
print(f"fpocket diagnostic probe on {probe_id}:")
_probe=run_fpocket(probe_id, verbose=True)
print(f"  -> {'OK, %d pockets' % len(_probe[0]) if _probe else 'FAILED'}\n")
if not _probe:
    raise RuntimeError("fpocket produced no output on the probe structure -- "
                       "see returncode/stderr above before running the full loop")

t0=time.time()
fp_results={}
for i,pid in enumerate(sorted(paths)):
    r=run_fpocket(pid)
    if r: fp_results[pid]=r
    if i%50==0: print(f"  fpocket {i}/{len(paths)} [{(time.time()-t0)/60:.1f}m]")
print(f"fpocket completed on {len(fp_results)}/{len(paths)} structures "
      f"in {(time.time()-t0)/60:.1f} min")

## Cell 4: Align, score, compare

In [ ]:
def residue_keys(pid, cid):
    """Ordered (chain,resnum,icode) using the SAME predicate as every other phase."""
    p=paths.get(pid)
    if p is None: return None
    try:
        with gzip.open(p,"rt") as fh:
            model_=next(PDBParser(QUIET=True,PERMISSIVE=True).get_structure(pid,fh).get_models())
        ch=model_[cid]
    except Exception: return None
    out=[]
    for r in ch:
        if r.get_resname().strip().upper() in AA3 and "CA" in r:
            out.append((cid, r.id[1], r.id[2].strip()))
    return out

rows=[]
skipped=0
for d in splits["test"]:
    key=(d.pdb_id,d.chain_id)
    if d.pdb_id not in fp_results: skipped+=1; continue
    keys=residue_keys(d.pdb_id,d.chain_id)
    if keys is None or len(keys)!=int(d.num_nodes): skipped+=1; continue
    pockets,scores=fp_results[d.pdb_id]
    if not pockets: skipped+=1; continue
    maxs=max(scores.values()) if scores else 1.0
    cont=np.zeros(len(keys),dtype=np.float32)
    top1=np.zeros(len(keys),dtype=int); top3=np.zeros(len(keys),dtype=int)
    ranks=sorted(pockets)
    for rk in ranks:
        sc=scores.get(rk,0.0)/max(maxs,1e-9)
        member=np.array([k in pockets[rk] for k in keys])
        cont=np.maximum(cont, member*sc)
        if rk==ranks[0]: top1|=member.astype(int)
        if rk in ranks[:3]: top3|=member.astype(int)
    rows.append({"y":d.y.numpy(),"model":model_pred[key],"fp":cont,"fp1":top1,"fp3":top3})

y=np.concatenate([r["y"] for r in rows]); mp=np.concatenate([r["model"] for r in rows])
fp=np.concatenate([r["fp"] for r in rows])
fp1=np.concatenate([r["fp1"] for r in rows]); fp3=np.concatenate([r["fp3"] for r in rows])
print(f"compared on {len(rows)} chains / {len(y):,} residues "
      f"({int(y.sum()):,} positives, {100*y.mean():.2f}%) | skipped {skipped} chains\n")

def prf(pred,y):
    tp=int(((pred==1)&(y==1)).sum()); 
    prec=tp/max(pred.sum(),1); rec=tp/max(y.sum(),1)
    return prec,rec,2*prec*rec/max(prec+rec,1e-9),matthews_corrcoef(y,pred)

print(f"{'method':<26}{'PR-AUC':>9}{'ROC-AUC':>10}{'prec':>8}{'rec':>8}{'F1':>8}{'MCC':>8}")
print("-"*77)
print(f"{'random (base rate)':<26}{y.mean():>9.4f}{0.5:>10.4f}{'-':>8}{'-':>8}{'-':>8}{'-':>8}")
print(f"{'fpocket (continuous)':<26}{average_precision_score(y,fp):>9.4f}"
      f"{roc_auc_score(y,fp):>10.4f}{'-':>8}{'-':>8}{'-':>8}{'-':>8}")
for nm,pr in (("fpocket top-1 pocket",fp1),("fpocket top-3 pockets",fp3)):
    p_,r_,f_,m_=prf(pr,y)
    print(f"{nm:<26}{'-':>9}{'-':>10}{p_:>8.3f}{r_:>8.3f}{f_:>8.3f}{m_:>8.3f}")
best_t,best_f1,best_m=0.5,0,0
for t in np.linspace(0.05,0.95,91):
    pd_=(mp>=t).astype(int)
    if pd_.sum():
        p_,r_,f_,m_=prf(pd_,y)
        if f_>best_f1: best_t,best_f1,best_m=t,f_,m_
p_,r_,f_,m_=prf((mp>=best_t).astype(int),y)
print(f"{'TopologicalEGNN':<26}{average_precision_score(y,mp):>9.4f}"
      f"{roc_auc_score(y,mp):>10.4f}{p_:>8.3f}{r_:>8.3f}{f_:>8.3f}{m_:>8.3f}")

# match fpocket top-3 recall, then compare precision at equal recall
tgt=fp3.sum()/len(fp3)
thr=np.quantile(mp,1-tgt)
p2,r2,f2,m2=prf((mp>=thr).astype(int),y)
print(f"\nat fpocket-top3 selectivity ({100*tgt:.1f}% of residues called positive):")
print(f"  fpocket top-3 : prec {prf(fp3,y)[0]:.3f} rec {prf(fp3,y)[1]:.3f} MCC {prf(fp3,y)[3]:.3f}")
print(f"  EGNN          : prec {p2:.3f} rec {r2:.3f} MCC {m2:.3f}")

json.dump({"n_chains":len(rows),"n_res":int(len(y)),
           "fpocket_pr":float(average_precision_score(y,fp)),
           "fpocket_roc":float(roc_auc_score(y,fp)),
           "egnn_pr":float(average_precision_score(y,mp)),
           "egnn_roc":float(roc_auc_score(y,mp))},
          open("/kaggle/working/phase7_baseline.json","w"),indent=2)
print("\nsaved -> phase7_baseline.json")

---

## Reading this

**fpocket uses no training data.** If `TopologicalEGNN` does not clearly beat it,
the learned model is not earning its complexity, and that would be the finding to
report.

The equal-selectivity block is the fairest single comparison: it forces both
methods to call the same *fraction* of residues positive, removing the advantage
a continuous score otherwise gets from threshold tuning.

One structural caveat: fpocket detects pockets from geometry alone and has no
notion of *which* pocket binds a ligand, so on multi-pocket proteins it is
penalised for finding real-but-unoccupied cavities. That is a genuine limitation
of the comparison, not a defect in fpocket, state it rather than claiming a
larger margin than the setup supports.